In [145]:
import pandas as pd
import numpy as np


In [146]:
patient = pd.read_csv('patients.csv')
treatments = pd.read_csv('treatments_full.csv')
adverse_reaction = pd.read_csv('adverse_reactions.csv')

### Issues with the dataset

1. Dirty Data
   
   Table - `Patients`

   - patient_id = 9 has miss spell name 'Dsvid' instead of David `accuracy`
   - state col sometimes contain full name and some time 2 word or 3 word `consistency`
   - some zip code col has entries with 4 digit `validity`
   - data missing for 12 patients in address,city, state,zip_code, country, contact `completion`
   - duplicate entries by the name of John Doe `accuracy`
   - one patient has weight = 48.8 pound `accuracy`
   - one patient has height = 27 inches `accuracy`

   Table - `Treatments`

    - given_name and surname col is is all lower case `consistency`
    - remove u from Auralin and Novadra cols `validity`
    - '-' in novadra and Auralin col treated as nan `validity`
    - missing values in hba1c_change col `completion`
    - 1 duplicate entry by the name Joseph day `accuracy`
    - in hba1c_change 9 instead of 4 `accuracy`

    Table - `Adverse_reactions`

    - given_name and surname are all in lower case `consistency`

2. Messy Data

   Table - `Patients`

      - contact col contains both phone and email

   Table - `Treatments`

      - Auralin and Novadra col should be split into 2 cols start and end dose
   

   Table - `Adverse_reactions`

      - This table should not exist independently


### Note - Assessing Data is an Iterative Process

### Data Quality Dimensions

- Completeness -> is data missing?
- Validity -> is data invalid -> negative height -> duplicate patient id
- Accuracy -> data is valid but not accurate -> weight -> 1kg
- Consistency -> both valid and accurate but written differently -> New Youk and NY


### Order of severity

Completeness <- Validity <- Accuracy <- Consistency

### Data Cleaning Order

1. Quality -> Completeness
2. Tidiness
3. Quality -> Validity
4. Quality -> Accuracy
5. Quality -> Consistency

#### Steps involved in Data cleaning
- Define
- Code
- Test

`Always make sure to create a copy of your pandas dataframe before you start the cleaning process`

In [147]:
patient_df = patient.copy()
treatments_df = treatments.copy()
adverse_reaction_df = adverse_reaction.copy()

### Define

- replace all missing values of patients df with no data
- sub hba1c_start from hba1c_end to get all the change values
- in patients table we will use regex to separate email and phone

In [148]:
# code 

patient_df['zip_code'] = patient_df['zip_code'].astype('str')


In [149]:
# code

patient_df.fillna('No Data',inplace=True)
patient_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 503 entries, 0 to 502
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   patient_id    503 non-null    int64  
 1   assigned_sex  503 non-null    str    
 2   given_name    503 non-null    str    
 3   surname       503 non-null    str    
 4   address       503 non-null    str    
 5   city          503 non-null    str    
 6   state         503 non-null    str    
 7   zip_code      503 non-null    str    
 8   country       503 non-null    str    
 9   contact       503 non-null    str    
 10  birthdate     503 non-null    str    
 11  weight        503 non-null    float64
 12  height        503 non-null    int64  
 13  bmi           503 non-null    float64
dtypes: float64(2), int64(2), str(10)
memory usage: 55.1 KB


In [150]:
treatments_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   given_name    350 non-null    str    
 1   surname       350 non-null    str    
 2   auralin       350 non-null    str    
 3   novodra       350 non-null    str    
 4   hba1c_start   350 non-null    float64
 5   hba1c_end     350 non-null    float64
 6   hba1c_change  213 non-null    float64
dtypes: float64(3), str(4)
memory usage: 19.3 KB


In [151]:
# code 

treatments_df['hba1c_change'] = treatments_df['hba1c_start'] - treatments_df['hba1c_end']

In [152]:
# test

treatments_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   given_name    350 non-null    str    
 1   surname       350 non-null    str    
 2   auralin       350 non-null    str    
 3   novodra       350 non-null    str    
 4   hba1c_start   350 non-null    float64
 5   hba1c_end     350 non-null    float64
 6   hba1c_change  350 non-null    float64
dtypes: float64(3), str(4)
memory usage: 19.3 KB


### Remove Messy data (Tidiness)

In [153]:
patient_df.head()

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,contact,birthdate,weight,height,bmi
0,1,female,Zoe,Wellish,576 Brown Bear Drive,Rancho California,California,92390.0,United States,951-719-9170ZoeWellish@superrito.com,7/10/1976,121.7,66,19.6
1,2,female,Pamela,Hill,2370 University Hill Road,Armstrong,Illinois,61812.0,United States,PamelaSHill@cuvox.de+1 (217) 569-3204,4/3/1967,118.8,66,19.2
2,3,male,Jae,Debord,1493 Poling Farm Road,York,Nebraska,68467.0,United States,402-363-6804JaeMDebord@gustr.com,2/19/1980,177.8,71,24.8
3,4,male,Liêm,Phan,2335 Webster Street,Woodbridge,NJ,7095.0,United States,PhanBaLiem@jourrapide.com+1 (732) 636-8246,7/26/1951,220.9,70,31.7
4,5,male,Tim,Neudorf,1428 Turkey Pen Lane,Dothan,AL,36303.0,United States,334-515-7487TimNeudorf@cuvox.de,2/18/1928,192.3,27,26.1


In [154]:
import re



# Initialize a list to collect the data
data = []

# Iterate through the data
for item in patient['contact']:
    # Convert item to a string to ensure compatibility with re.search
    item = str(item)

    # Use regular expressions to find the phone numbers and email addresses
    phone_match = re.search(r'(\d{3}[-\.\s]??\d{3}[-\.\s]??\d{4}|\(\d{3}\)\s*\d{3}[-\.\s]??\d{4}|\d{3}[-\.\s]??\d{4})', item)
    phone = phone_match.group(0) if phone_match else None

    # Remove the phone number from the item
    item = re.sub(r'(\d{3}[-\.\s]??\d{3}[-\.\s]??\d{4}|\(\d{3}\)\s*\d{3}[-\.\s]??\d{4}|\d{3}[-\.\s]??\d{4})', '', item)

    email_match = re.search(r'([a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+)', item)
    email = email_match.group(0) if email_match else None

    data.append({'phone': phone, 'email': email})

# Create a DataFrame from the collected data
df = pd.DataFrame(data, columns=['phone', 'email'])

print(df)

              phone                               email
0      951-719-9170            ZoeWellish@superrito.com
1    (217) 569-3204                PamelaSHill@cuvox.de
2      402-363-6804                JaeMDebord@gustr.com
3    (732) 636-8246           PhanBaLiem@jourrapide.com
4      334-515-7487                 TimNeudorf@cuvox.de
..              ...                                 ...
498    207-477-0579     MustafaLindstrom@jourrapide.com
499    928-284-4492              RumanBisliev@gustr.com
500    816-223-6007           JinkedeKeizer@teleworm.us
501    360 443 2060  ChidaluOnyekaozulu@jourrapide.com1
502    402-848-4923            PatrickGersten@rhyta.com

[503 rows x 2 columns]


In [155]:
patient_df['phone'] = df['phone']
patient_df['email'] = df['email']

In [156]:
patient_df.drop(columns='contact',inplace=True)

In [157]:
patient_df.sample()

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,birthdate,weight,height,bmi,phone,email
307,308,female,Mathea,Lillebø,4168 Coventry Court,Gulfport,MS,39501.0,United States,11/30/1928,112.4,65,18.7,228-237-2271,MatheaLilleb@fleckens.hu


In [194]:
# validity Issue: zip code col has entries with 4 digit

patient_df['zip_code'].str.len().value_counts()

zip_code
7    449
6     49
Name: count, dtype: int64

In [200]:
patient_df.zip_code.sample(5)

75     94538.0
435    23601.0
387    33301.0
78      2048.0
396    33870.0
Name: zip_code, dtype: str

In [202]:
# Data is in string but decimal format -  Need to remove decimal part and fill zeros upfrom to make it in 5 digit format
# Function to clean the zip code
def clean_zip_code(zip_code):
    if zip_code == 'No Data':
        return 'NA'
    zip_code = int(float(zip_code))  # Remove decimal part
    return str(zip_code).zfill(5)    # Convert to string and pad with zeros

# Apply the function to the zip_code column
patient_df['zip_code'].apply(clean_zip_code).str.len().value_counts()

zip_code
5    486
2     12
Name: count, dtype: int64

In [203]:
# Apply the function to the zip_code column
patient_df['zip_code'] = patient_df['zip_code'].apply(clean_zip_code)

In [204]:
# There is an issue: data missing for 12 patients in address,city, state,zip_code ,country, contact completion

patient_df[patient_df.isnull().any(axis=1)]

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,birthdate,weight,height,bmi,phone,email
209,210,female,Lalita,Eldarkhanov,No Data,No Data,No Data,NA,No Data,8/14/1950,143.4,62,26.2,NaN,NaN
219,220,male,Mỹ,Quynh,No Data,No Data,No Data,NA,No Data,4/9/1978,237.8,69,35.1,NaN,NaN
230,231,female,Elisabeth,Knudsen,No Data,No Data,No Data,NA,No Data,9/23/1976,165.9,63,29.4,NaN,NaN
234,235,female,Martina,Tománková,No Data,No Data,No Data,NA,No Data,4/7/1936,199.5,65,33.2,NaN,NaN
242,243,male,John,O'Brian,No Data,No Data,No Data,NA,No Data,2/25/1957,205.3,74,26.4,NaN,NaN
249,250,male,Benjamin,Mehler,No Data,No Data,No Data,NA,No Data,10/30/1951,146.5,69,21.6,NaN,NaN
257,258,male,Jin,Kung,No Data,No Data,No Data,NA,No Data,5/17/1995,231.7,69,34.2,NaN,NaN
264,265,female,Wafiyyah,Asfour,No Data,No Data,No Data,NA,No Data,11/3/1989,158.6,63,28.1,NaN,NaN
269,270,female,Flavia,Fiorentino,No Data,No Data,No Data,NA,No Data,10/9/1937,175.2,61,33.1,NaN,NaN
278,279,female,Generosa,Cabán,No Data,No Data,No Data,NA,No Data,12/16/1962,124.3,69,18.4,NaN,NaN


In [214]:
patient_df[['address', 'city', 'state', 'zip_code', 'country', 'phone','email']] = patient_df[['address', 'city', 'state', 'zip_code', 'country', 'phone','email']].fillna('Unknown')
patient_df[patient_df.isnull().any(axis=1)]

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,birthdate,weight,height,bmi,phone,email


In [215]:
# Validity Issue : incorrect data type assigned to sex, zip code, birthdate 

patient_df.info()

<class 'pandas.DataFrame'>
Index: 498 entries, 0 to 502
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   patient_id    498 non-null    int64         
 1   assigned_sex  498 non-null    category      
 2   given_name    498 non-null    str           
 3   surname       498 non-null    str           
 4   address       498 non-null    str           
 5   city          498 non-null    str           
 6   state         498 non-null    str           
 7   zip_code      498 non-null    str           
 8   country       498 non-null    str           
 9   birthdate     498 non-null    datetime64[us]
 10  weight        498 non-null    float64       
 11  height        498 non-null    int64         
 12  bmi           498 non-null    float64       
 13  phone         498 non-null    str           
 14  email         498 non-null    str           
dtypes: category(1), datetime64[us](1), float64(2), int64(2),

In [216]:
patient_df['assigned_sex'] = patient_df['assigned_sex'].astype('category') # Earlier Object
patient_df['zip_code'] = patient_df['zip_code'].astype(str) # Already Corrected while making it for 5 digit
patient_df['birthdate'] = pd.to_datetime(patient_df['birthdate'], errors='coerce') # Earlier Object
patient_df.info()

<class 'pandas.DataFrame'>
Index: 498 entries, 0 to 502
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   patient_id    498 non-null    int64         
 1   assigned_sex  498 non-null    category      
 2   given_name    498 non-null    str           
 3   surname       498 non-null    str           
 4   address       498 non-null    str           
 5   city          498 non-null    str           
 6   state         498 non-null    str           
 7   zip_code      498 non-null    str           
 8   country       498 non-null    str           
 9   birthdate     498 non-null    datetime64[us]
 10  weight        498 non-null    float64       
 11  height        498 non-null    int64         
 12  bmi           498 non-null    float64       
 13  phone         498 non-null    str           
 14  email         498 non-null    str           
dtypes: category(1), datetime64[us](1), float64(2), int64(2),

In [217]:
#  `consistency` Issue : state col sometimes contain full name and some time 2 word or 3 word 

patient_df.state.value_counts()

state
CA         60
NY         42
TX         32
IL         24
FL         22
MA         22
PA         18
GA         15
OH         14
MI         13
OK         13
LA         13
NJ         12
Unknown    12
VA         11
WI         10
MS         10
AL          9
TN          9
MN          9
IN          9
NC          8
KY          8
WA          8
MO          7
NE          6
NV          6
KS          6
ID          6
IA          5
CT          5
SC          5
CO          4
ME          4
AZ          4
ND          4
RI          4
AR          4
SD          3
DE          3
MD          3
WV          3
OR          3
MT          2
VT          2
DC          2
NM          1
WY          1
AK          1
NH          1
Name: count, dtype: int64

In [211]:
state_abbreviations = {
    'California': 'CA',
    'Texas': 'TX',
    'New York': 'NY',
    'Massachusetts': 'MA',
    'Pennsylvania': 'PA',
    'Georgia': 'GA',
    'Illinois': 'IL',
    'Ohio': 'OH',
    'Florida': 'FL',
    'Michigan': 'MI',
    'Oklahoma': 'OK',
    'Louisiana': 'LA',
    'New Jersey': 'NJ',
    'Virginia': 'VA',
    'Mississippi': 'MS',
    'Wisconsin': 'WI',
    'Indiana': 'IN',
    'Minnesota': 'MN',
    'Tennessee': 'TN',
    'Alabama': 'AL',
    'North Carolina': 'NC',
    'Kentucky': 'KY',
    'Washington': 'WA',
    'Missouri': 'MO',
    'Idaho': 'ID',
    'Kansas': 'KS',
    'Nevada': 'NV',
    'South Carolina': 'SC',
    'Iowa': 'IA',
    'Connecticut': 'CT',
    'Maine': 'ME',
    'North Dakota': 'ND',
    'Nebraska': 'NE',
    'Rhode Island': 'RI',
    'Arkansas': 'AR',
    'Colorado': 'CO',
    'Arizona': 'AZ',
    'Maryland': 'MD',
    'Delaware': 'DE',
    'West Virginia': 'WV',
    'Oregon': 'OR',
    'South Dakota': 'SD',
    'Montana': 'MT',
    'Vermont': 'VT',
    'District of Columbia': 'DC',
    'Alaska': 'AK',
    'Wyoming': 'WY',
    'New Hampshire': 'NH',
    'New Mexico': 'NM',
    'No Data': 'Unknown'
}

# Apply changes to the DataFrame
patient_df['state'] = patient_df['state'].apply(lambda x: state_abbreviations.get(x, x))

# Final check of the dataframe
print(patient_df['state'].value_counts())


state
CA         60
NY         42
TX         32
IL         24
FL         22
MA         22
PA         18
GA         15
OH         14
MI         13
OK         13
LA         13
NJ         12
Unknown    12
VA         11
WI         10
MS         10
AL          9
TN          9
MN          9
IN          9
NC          8
KY          8
WA          8
MO          7
NE          6
NV          6
KS          6
ID          6
IA          5
CT          5
SC          5
CO          4
ME          4
AZ          4
ND          4
RI          4
AR          4
SD          3
DE          3
MD          3
WV          3
OR          3
MT          2
VT          2
DC          2
NM          1
WY          1
AK          1
NH          1
Name: count, dtype: int64


In [218]:
patient_df.info()

<class 'pandas.DataFrame'>
Index: 498 entries, 0 to 502
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   patient_id    498 non-null    int64         
 1   assigned_sex  498 non-null    category      
 2   given_name    498 non-null    str           
 3   surname       498 non-null    str           
 4   address       498 non-null    str           
 5   city          498 non-null    str           
 6   state         498 non-null    str           
 7   zip_code      498 non-null    str           
 8   country       498 non-null    str           
 9   birthdate     498 non-null    datetime64[us]
 10  weight        498 non-null    float64       
 11  height        498 non-null    int64         
 12  bmi           498 non-null    float64       
 13  phone         498 non-null    str           
 14  email         498 non-null    str           
dtypes: category(1), datetime64[us](1), float64(2), int64(2),

In [ ]:
patient_df.to_csv('Cleaned_patient_data.csv',index=False)




In [221]:
clean_data = pd.read_csv('Cleaned_patient_data.csv')
clean_data.head()

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,birthdate,weight,height,bmi,phone,email
0,1,female,Zoe,Wellish,576 Brown Bear Drive,Rancho California,CA,92390.0,United States,1976-07-10,121.7,66,19.6,951-719-9170,ZoeWellish@superrito.com
1,2,female,Pamela,Hill,2370 University Hill Road,Armstrong,IL,61812.0,United States,1967-04-03,118.8,66,19.2,(217) 569-3204,PamelaSHill@cuvox.de
2,3,male,Jae,Debord,1493 Poling Farm Road,York,NE,68467.0,United States,1980-02-19,177.8,71,24.8,402-363-6804,JaeMDebord@gustr.com
3,4,male,Liêm,Phan,2335 Webster Street,Woodbridge,NJ,7095.0,United States,1951-07-26,220.9,70,31.7,(732) 636-8246,PhanBaLiem@jourrapide.com
4,71,male,Tim,Neudorf,1428 Turkey Pen Lane,Dothan,AL,36303.0,United States,1928-02-18,192.3,72,26.1,334-515-7487,TimNeudorf@cuvox.de


In [177]:
# `accuracy` Issue 1: patient_id = 9 has miss spell name 'Dsvid' instead of David 

patient_df.loc[patient_df['patient_id'] == 9, 'given_name']

8    Dsvid
Name: given_name, dtype: str

In [178]:
# Correct Misspelled

patient_df.loc[patient_df['patient_id'] == 9, 'given_name'] = 'David'
patient_df.loc[patient_df['patient_id'] == 9, 'given_name']


8    David
Name: given_name, dtype: str

In [179]:
# accuary issue 2: one patient has height = 27 inches

patient_df.loc[(patient_df['height'] == 27)]

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,birthdate,weight,height,bmi,phone,email
4,5,male,Tim,Neudorf,1428 Turkey Pen Lane,Dothan,AL,36303.0,United States,2/18/1928,192.3,27,26.1,334-515-7487,TimNeudorf@cuvox.de


In [180]:
# Given values
weight_lb = 192.3
bmi = 26.1

# Convert weight from pounds to kilograms
weight_kg = weight_lb * 0.453592

# Calculate height in meters
height_m = (weight_kg / bmi)**0.5

# Convert height from meters to inches
height_inches = height_m * 39.3701

print(f"Corrected height in inches: {height_inches:.2f}")


Corrected height in inches: 71.97


In [184]:
# Corrected this issue

patient_df.loc[patient_df['height'] == 27, 'height' ] = 72
patient_df.loc[(patient_df['height'] == 27)]

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,birthdate,weight,height,bmi,phone,email


In [186]:
# accuary issue 3: one patient has weight = 48.8 inches

patient_df.loc[(patient_df['weight'] == 48.8)]


,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,birthdate,weight,height,bmi,phone,email
210,211,female,Camilla,Zaitseva,4689 Briarhill Lane,Wooster,OH,44691.0,United States,11/26/1938,48.8,63,19.1,330-202-2145,CamillaZaitseva@superrito.com


In [187]:
# Given Values
height_inches = 63
bmi = 19.1

# calculate weight:
weight_lb = (bmi * (height_inches / 39.3701)**2) / 0.453592

print(f'weight in pounds:{weight_lb:.2f}')

weight in pounds:107.82


In [189]:
# Corrected this weight

patient_df.loc[patient_df['weight'] == 48.8, 'weight'] = 107.82
patient_df.loc[(patient_df['weight'] == 48.8)]

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,birthdate,weight,height,bmi,phone,email


In [190]:
# Accuracy Issue 3: duplicate entries by the name of John Doe

patient_df[patient_df.duplicated(['given_name','surname'])]

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,birthdate,weight,height,bmi,phone,email
229,230,male,John,Doe,123 Main Street,New York,NY,12345.0,United States,1/1/1975,180.0,72,24.4,1234567890,johndoe@email.com
237,238,male,John,Doe,123 Main Street,New York,NY,12345.0,United States,1/1/1975,180.0,72,24.4,1234567890,johndoe@email.com
244,245,male,John,Doe,123 Main Street,New York,NY,12345.0,United States,1/1/1975,180.0,72,24.4,1234567890,johndoe@email.com
251,252,male,John,Doe,123 Main Street,New York,NY,12345.0,United States,1/1/1975,180.0,72,24.4,1234567890,johndoe@email.com
277,278,male,John,Doe,123 Main Street,New York,NY,12345.0,United States,1/1/1975,180.0,72,24.4,1234567890,johndoe@email.com


In [193]:
# Remove duplicates entries

patient_df = patient_df.drop_duplicates(subset=['given_name','surname'],keep='first')
patient_df[patient_df.duplicated(subset=['given_name','surname'])]

,patient_id,assigned_sex,given_name,surname,address,city,state,zip_code,country,birthdate,weight,height,bmi,phone,email


In [159]:
# trearment table

treatments_df = treatments_df.melt(id_vars=['given_name','surname','hba1c_start','hba1c_end','hba1c_change'],var_name='Dosage',value_name='Dosage_range')


In [160]:
treatments_df = treatments_df[treatments_df['Dosage_range'] != '-']

In [161]:
treatments_df['Dosage_start'] = treatments_df['Dosage_range'].str.split('-').str.get(0)
treatments_df['Dosage_end'] = treatments_df['Dosage_range'].str.split('-').str.get(1)

In [162]:
treatments_df.drop(columns='Dosage_range',inplace=True)

In [163]:
treatments_df.head()

,given_name,surname,hba1c_start,hba1c_end,hba1c_change,Dosage,Dosage_start,Dosage_end
0,veronika,jindrová,7.63,7.20,0.43,auralin,41u,48u
3,skye,gormanston,7.97,7.62,0.35,auralin,33u,36u
6,sophia,haugen,7.65,7.27,0.38,auralin,37u,42u
7,eddie,archer,7.89,7.55,0.34,auralin,31u,38u
9,asia,woźniak,7.76,7.37,0.39,auralin,30u,36u


In [164]:
treatments_df['Dosage_start'] = treatments_df['Dosage_start'].str.replace('u','')
treatments_df['Dosage_end'] = treatments_df['Dosage_end'].str.replace('u','')

In [165]:
treatments_df['Dosage_start'] = treatments_df['Dosage_start'].astype('int')
treatments_df['Dosage_end'] = treatments_df['Dosage_end'].astype('int')

In [166]:
treatments_df.head()


,given_name,surname,hba1c_start,hba1c_end,hba1c_change,Dosage,Dosage_start,Dosage_end
0,veronika,jindrová,7.63,7.20,0.43,auralin,41,48
3,skye,gormanston,7.97,7.62,0.35,auralin,33,36
6,sophia,haugen,7.65,7.27,0.38,auralin,37,42
7,eddie,archer,7.89,7.55,0.34,auralin,31,38
9,asia,woźniak,7.76,7.37,0.39,auralin,30,36


In [167]:
treatments_df.info()

<class 'pandas.DataFrame'>
Index: 350 entries, 0 to 698
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   given_name    350 non-null    str    
 1   surname       350 non-null    str    
 2   hba1c_start   350 non-null    float64
 3   hba1c_end     350 non-null    float64
 4   hba1c_change  350 non-null    float64
 5   Dosage        350 non-null    str    
 6   Dosage_start  350 non-null    int64  
 7   Dosage_end    350 non-null    int64  
dtypes: float64(3), int64(2), str(3)
memory usage: 24.6 KB


In [168]:
# merge treatment table and adverse_reaction

treatments_df = treatments_df.merge(adverse_reaction_df,how='left',on=['given_name','surname'])

In [169]:
treatments_df

,given_name,surname,hba1c_start,hba1c_end,hba1c_change,Dosage,Dosage_start,Dosage_end,adverse_reaction
0,veronika,jindrová,7.63,7.20,0.43,auralin,41,48,NaN
1,skye,gormanston,7.97,7.62,0.35,auralin,33,36,NaN
2,sophia,haugen,7.65,7.27,0.38,auralin,37,42,NaN
3,eddie,archer,7.89,7.55,0.34,auralin,31,38,NaN
4,asia,woźniak,7.76,7.37,0.39,auralin,30,36,NaN
...,...,...,...,...,...,...,...,...,...
345,christopher,woodward,7.51,7.06,0.45,novodra,55,51,nausea
346,maret,sultygov,7.67,7.30,0.37,novodra,26,23,NaN
347,lixue,hsueh,9.21,8.80,0.41,novodra,22,23,injection site discomfort
348,jakob,jakobsen,7.96,7.51,0.45,novodra,28,26,hypoglycemia


In [170]:
treatments_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   given_name        350 non-null    str    
 1   surname           350 non-null    str    
 2   hba1c_start       350 non-null    float64
 3   hba1c_end         350 non-null    float64
 4   hba1c_change      350 non-null    float64
 5   Dosage            350 non-null    str    
 6   Dosage_start      350 non-null    int64  
 7   Dosage_end        350 non-null    int64  
 8   adverse_reaction  35 non-null     str    
dtypes: float64(3), int64(2), str(4)
memory usage: 24.7 KB


In [171]:
# `consistency` issue :given_name and surname are all in lower case 

treatments_df['given_name'] = treatments_df['given_name'].str.capitalize()
treatments_df['surname'] = treatments_df['surname'].str.capitalize()

In [172]:
treatments_df.sample(5)

,given_name,surname,hba1c_start,hba1c_end,hba1c_change,Dosage,Dosage_start,Dosage_end,adverse_reaction
127,Alex,Crawford,7.69,7.30,0.39,auralin,51,62,hypoglycemia
186,Chiemela,Tobeolisa,7.59,7.17,0.42,novodra,43,47,NaN
94,Chân,Bùi,7.53,7.18,0.35,auralin,31,42,NaN
25,Oles,Zhdanov,7.52,7.11,0.41,auralin,54,67,NaN
16,Alexander,Mathiesen,7.96,7.55,0.41,auralin,47,58,NaN


In [173]:
# `accuracy` issue 1:  1 duplicate entry by the name Joseph day

treatments_df[treatments_df.duplicated(subset=['given_name','surname'])] 

,given_name,surname,hba1c_start,hba1c_end,hba1c_change,Dosage,Dosage_start,Dosage_end,adverse_reaction
62,Joseph,Day,7.7,7.19,0.51,auralin,29,36,hypoglycemia


In [174]:
# Remove the duplicate row 

treatments_df = treatments_df.drop_duplicates(subset=['given_name','surname'], keep='first')


In [175]:
treatments_df[treatments_df['given_name']=='Joseph']

,given_name,surname,hba1c_start,hba1c_end,hba1c_change,Dosage,Dosage_start,Dosage_end,adverse_reaction
5,Joseph,Day,7.70,7.19,0.51,auralin,29,36,hypoglycemia
167,Joseph,Tucker,7.67,7.30,0.37,auralin,48,56,NaN


In [176]:
# accuracy issue 2: in hba1c_change 9 instead of 4 

treatments_df[treatments_df.hba1c_change==9]  # Aready Treasted



,given_name,surname,hba1c_start,hba1c_end,hba1c_change,Dosage,Dosage_start,Dosage_end,adverse_reaction


In [222]:
treatments_df.info()

<class 'pandas.DataFrame'>
Index: 349 entries, 0 to 349
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   given_name        349 non-null    str    
 1   surname           349 non-null    str    
 2   hba1c_start       349 non-null    float64
 3   hba1c_end         349 non-null    float64
 4   hba1c_change      349 non-null    float64
 5   Dosage            349 non-null    str    
 6   Dosage_start      349 non-null    int64  
 7   Dosage_end        349 non-null    int64  
 8   adverse_reaction  34 non-null     str    
dtypes: float64(3), int64(2), str(4)
memory usage: 27.3 KB


In [223]:
treatments_df.to_csv('Cleaned_Treatment_data.csv',index=False)